# Экспертиза записей Chatterbox на слух: Паоло (it, ru) и Хава (he), gemini-3.1-flash-live-preview

Вход — последний прогон из `results/` репозитория (или вывод `italo-tutor-chatterbox-eval`, подключённый как Input). Ключ Gemini — Kaggle Secret `GEMINI_API_KEY`
(Add-ons → Secrets, галочка для этого ноутбука). GPU не нужен, Internet = On.
Вопросы слепые: эксперт выбирает из вариантов в случайном порядке, что прозвучало; ответ сравнивается с задуманным.

In [ ]:
!pip install -q websockets soundfile
!rm -rf /kaggle/working/task && git clone -q --depth 1 https://github.com/vasiliad/italo-tutor-chatterbox /kaggle/working/task

In [ ]:
import os, glob, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient
# Ключи: секреты GEMINI_API_KEY, GEMINI_API_KEY_2 … _9 (каждый отдельным секретом с галочкой)
keys, sc = [], UserSecretsClient()
for name in ["GEMINI_API_KEY"] + [f"GEMINI_API_KEY_{i}" for i in range(2, 10)]:
    try:
        keys.append(sc.get_secret(name).strip())
    except Exception:
        pass
print("ключей:", len(keys))
assert keys, "Нет секрета GEMINI_API_KEY"
os.makedirs(os.path.expanduser("~/key"), exist_ok=True)
open(os.path.expanduser("~/key/key"), "w").write("\n".join(keys))

# Записи: подключённый Input или последний прогон в репозитории (results/<дата>_<версия>/)
src = glob.glob("/kaggle/input/**/results.csv", recursive=True) or \
      sorted(glob.glob("/kaggle/working/task/results/*/results.csv"))[-1:]
print("найдено:", src)
assert src, "Нет записей: ни Input, ни results/ в репозитории"
ROOT = "/kaggle/working/chatterbox_eval"
shutil.copytree(os.path.dirname(src[0]), ROOT, dirs_exist_ok=True)

In [ ]:
# Сначала проверка связи: 3 задания
!cd /kaggle/working/task/tools/expert && python3 expert_listen.py --root /kaggle/working/chatterbox_eval --limit 3

In [ ]:
# Полная экспертиза: не чаще 20 вопросов в минуту на ключ (ключи из разных проектов — лимиты у каждого свои), по приоритету.
# Продолжает с места остановки: ответы прошлых запусков берутся из репозитория (expert_results.csv).
!cd /kaggle/working/task/tools/expert && python3 expert_listen.py --root /kaggle/working/chatterbox_eval --per-min 20 --per-key

In [ ]:
import pandas as pd
df = pd.read_csv("/kaggle/working/chatterbox_eval/expert_results.csv")
print(open("/kaggle/working/chatterbox_eval/expert_summary.md").read())
out = "/kaggle/working/expert_out"
os.makedirs(out, exist_ok=True)
for f in ["expert_results.csv", "expert_summary.md"]:
    shutil.copy(f"/kaggle/working/chatterbox_eval/{f}", out)
shutil.rmtree("/kaggle/working/chatterbox_eval"); shutil.rmtree("/kaggle/working/task", ignore_errors=True)
os.remove(os.path.expanduser("~/key/key"))